# 🧩 numpy — the puzzle on small data

**Time:** ~40 minutes &nbsp;•&nbsp; **Goal:** learn every step of the lecture's neuron puzzle on data small enough to **check by eye** — so that the real analysis, in notebook 2, is just the same steps on a bigger array.

## The puzzle

In the lecture you met a real neuron, recorded while a mouse felt two whisker "words": **go** (lick for water) and **nogo** (don't lick). The question was: **does the neuron tell them apart?**

The real file has 21,238 frames — far too many to look at. So here is a **made-up neuron with 50 frames**. It behaves like the real one, but you can see every number:

- each word lasts **3 frames** (in the real data: 9)
- there are **3 go** and **3 nogo** trials (real: 254 and 264)

Each step below is one move in the analysis. Same rhythm as Lab 4: 📘 **Demo** — 🔬 **Write it** with a **self-check** — 🐞 **Broken**.

1. Meet the arrays — attributes and methods
2. A first try: a mask
3. Where is the word on? — `np.where`
4. Where does each trial start? — a slice with a step
5. Cut one window round a start
6. Cut them all, and stack them into a grid
7. Average down the columns — `axis=0`
8. go against nogo — the answer

## Our data — a neuron you can read

Run this first. Every exercise below uses these three arrays, one number per frame:

- `dff` — the neuron's activity (around 0 = resting)
- `go` — `1` while the go word is playing, `0` otherwise
- `nogo` — the same for the nogo word

The cell prints the first 16 frames lined up. Look at frames 9–13: the go word plays (9, 10, 11) — and then the neuron jumps (12, 13).

In [ ]:
# Run this first. A made-up neuron, 50 frames long - small enough to read.
import numpy as np

dff = np.array([0.0, 0.1, -0.1, 0.0, 0.2, 0.1, 0.0, -0.1, 0.1, 0.0,     # frames  0-9
                0.1, -0.1, 0.9, 1.2, 0.4, 0.1, 0.0, 0.1, 0.0, -0.1,     # frames 10-19
                0.3, 0.0, 0.1, 0.0, -0.1, 0.1, 0.0, -0.1, 0.8, 1.0,     # frames 20-29
                0.5, 0.0, 0.1, 0.0, -0.1, 0.1, 0.1, 0.0, -0.1, 0.0,     # frames 30-39
                0.1, -0.1, 0.1, 0.0, 1.0, 0.8, 0.3, 0.1, 0.0, -0.1])    # frames 40-49

go = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
               1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
               0, 0, 0, 0, 0, 1, 1, 1, 0, 0,
               0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
               0, 1, 1, 1, 0, 0, 0, 0, 0, 0])

nogo = np.array([0, 1, 1, 1, 0, 0, 0, 0, 0, 0,
                 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
                 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                 0, 0, 0, 1, 1, 1, 0, 0, 0, 0,
                 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

print("frame:", np.arange(16))
print("dff:  ", dff[:16])
print("go:   ", go[:16])
print("nogo: ", nogo[:16])

## Step 1 — Meet the arrays

An array is an object with a reception (lecture 5): **attributes** are facts it knows — no brackets; **methods** are jobs it does — brackets.

```python
dff.shape      # attribute: (50,) — 50 numbers in one row
dff.mean()     # method: the average
dff.max()      # the largest value
dff.argmax()   # WHERE the largest value is — its position
```

And arithmetic happens to every value at once: `dff * 100` multiplies all 50.

In [ ]:
# DEMO - ask the array about itself
print("shape:", dff.shape)
print("mean: ", round(dff.mean(), 3))
print("max:  ", dff.max(), "at frame", dff.argmax())
print("x 100:", (dff * 100)[:6])        # every value, at once

### 🔬 Write it

**Your task:**

1. Store how many frames there are in `n_frames` — use an attribute.
2. Store the **lowest** value of `dff` in `lowest`, and **where** it is in `lowest_frame`.

> 💡 The partners of `.max()` and `.argmax()` are `.min()` and `.argmin()`. Check your answer against the printed frames above.

In [ ]:
# TODO — n_frames, lowest and lowest_frame


# --- self-check (do not edit) ---
assert "n_frames" in globals(), "❌ No variable called n_frames yet"
assert "lowest" in globals() and "lowest_frame" in globals(), "❌ Make lowest and lowest_frame"
assert n_frames == 50, f"❌ n_frames should be 50, got {n_frames} — dff.size"
assert lowest == -0.1, f"❌ lowest should be -0.1, got {lowest} — dff.min()"
assert lowest_frame != -0.1, "❌ lowest_frame is the lowest VALUE — you want its POSITION: dff.argmin()"
assert lowest_frame == 2, f"❌ lowest_frame should be 2, got {lowest_frame} — dff.argmin() gives the FIRST position of the lowest value"
print("✅ Correct — and notice argmin gives the FIRST frame with -0.1: there are several")

### 🐞 Broken — the brackets matter

Run it, read the **last line** of the error, then fix it. It's two characters.

In [ ]:
# 🐞 BROKEN — the peak, rounded to 1 decimal
peak = dff.max
print("peak:", round(peak, 1))


# --- self-check (do not edit) ---
assert not callable(peak), "❌ peak is still the method itself — add brackets to run it: dff.max()"
assert peak == 1.2, f"❌ peak should be 1.2, got {peak}"
print("✅ Fixed — dff.max is the method; dff.max() is its answer")

## Step 2 — A first try: a mask

The obvious first question: what does the neuron do **while** the go word plays?

`go == 1` compares every value at once and gives a **mask** — `True` where go is on. Put the mask in square brackets and only the `True` frames are kept:

```python
mask = go == 1
dff[mask]          # only the frames where go is on
```

In [ ]:
# DEMO - activity while go is on
mask = go == 1
print(mask[:16])
print("dff while go is on:", dff[mask])
print("their mean:", round(dff[mask].mean(), 2))
print("mean overall:", round(dff.mean(), 2))

### 🔬 Write it

**Your task:** store the mean of `dff` **while nogo is on**, rounded to 2 decimal places, in `nogo_on_mean`.

In [ ]:
# TODO — the mean of dff while nogo is on, rounded to 2 decimals


# --- self-check (do not edit) ---
assert "nogo_on_mean" in globals(), "❌ No variable called nogo_on_mean yet"
assert nogo_on_mean != round(dff.mean(), 2), "❌ That's the mean of ALL frames — filter with the mask first: dff[nogo == 1]"
assert nogo_on_mean == 0.0, f"❌ nogo_on_mean should be 0.0, got {nogo_on_mean} — dff[nogo == 1].mean(), rounded to 2"
print("✅ Correct — 0.0 for both words. The mask finds nothing!")

**Both words give 0.0** — yet the printout shows the neuron jumping to 0.9 and 1.2 at frames 12 and 13. The mask missed it because the neuron answers **after** the word, not while it plays.

That is exactly what happened in the lecture with the real neuron. So we need to look at each trial as a little movie — a bit **before** the word, the word, and what comes **after**. First: where does each word start?

## Step 3 — Where is the word on? `np.where`

The mask says *whether* each frame is on. `np.where` turns it into **positions** — the frame numbers where it is `True`:

```python
np.where(go == 1)[0]
```

`np.where` hands back a **tuple** with one array inside; `[0]` takes the array out.

In [ ]:
# DEMO - the frame numbers where go is on
go_on = np.where(go == 1)[0]
print(go_on)

Three blocks of three: `9 10 11`, `25 26 27`, `41 42 43` — one block per trial.

### 🐞 Broken — the tuple trap

This should count the frames where **nogo** is on — that's 9. No error, just a wrong answer.

> 💡 Print `nogo_on` itself. What is the outer thing, and how many items does it hold?

In [ ]:
# 🐞 BROKEN — every frame where nogo is on (there are 9)
nogo_on = np.where(nogo == 1)
print(len(nogo_on), "frames")


# --- self-check (do not edit) ---
assert len(nogo_on) == 9, f"❌ len(nogo_on) is {len(nogo_on)} — np.where hands back a TUPLE with one array inside. Take it out with [0]"
assert list(nogo_on) == [1, 2, 3, 17, 18, 19, 33, 34, 35], f"❌ nogo_on should be [1 2 3 17 18 19 33 34 35], got {nogo_on}"
print("✅ Fixed —", nogo_on)

## Step 4 — Where does each trial start?

We only want the **first** frame of each block: `9`, `25`, `41`.

Every word lasts exactly 3 frames, so the starts are every **third** position in `go_on`. A slice with a **step** does that — the same as on a list:

```python
go_on[::3]      # from the start, to the end, in steps of 3
```

> In the real data every word lasts **9** frames — so there it's `[::9]`.

In [ ]:
# DEMO - the first frame of each go trial
go_starts = go_on[::3]
print(go_starts)
print(len(go_starts), "go trials")

### 🔬 Write it

**Your task:** make `nogo_starts` — the first frame of each **nogo** trial — from `nogo_on`.

Then look at the first one. Remember it: it will matter in Step 6.

In [ ]:
# TODO — nogo_starts, from nogo_on


# --- self-check (do not edit) ---
assert "nogo_starts" in globals(), "❌ No variable called nogo_starts yet"
assert len(nogo_starts) != 9, "❌ That's every nogo frame — keep every third with [::3]"
assert list(nogo_starts) == [1, 17, 33], f"❌ nogo_starts should be [1 17 33], got {nogo_starts}"
print("✅ Correct — the first nogo trial starts at frame 1, right at the beginning of the recording")

## Step 5 — Cut one window round a start

For each trial we cut the same **window**: 2 frames **before** the start, and 5 from the start on — 7 frames:

```python
start = go_starts[0]            # 9
dff[start - 2 : start + 5]      # frames 7, 8, ... 13
```

In the window, the word starts at **position 2**. Anything before it is the **baseline** — what the neuron did before the word.

> Real data: 10 frames before, 30 after → `dff[start - 10 : start + 30]`.

In [ ]:
# DEMO - the window round the first go trial
start = go_starts[0]
window = dff[start - 2 : start + 5]
print("frames:", np.arange(start - 2, start + 5))
print("dff:   ", window)

### 🔬 Write it

**Your task:** cut the window round the **second** go trial and store it in `second_window`.

> 💡 The second start is `go_starts[1]`. Check your answer by eye against the data cell: which frames should it be?

In [ ]:
# TODO — the window round the second go trial


# --- self-check (do not edit) ---
assert "second_window" in globals(), "❌ No variable called second_window yet"
assert len(second_window) == 7, f"❌ A window should be 7 frames, got {len(second_window)} — dff[start - 2 : start + 5]"
assert np.allclose(second_window, dff[23:30]), f"❌ second_window should be frames 23 to 29: {dff[23:30]}, got {second_window} — use go_starts[1]"
print("✅ Correct — and there's the jump again, 3 frames after the start:", second_window)

## Step 6 — Cut them all, and stack them into a grid

Now do Step 5 for **every** start with a loop, and collect the windows in a list. `np.array()` then **stacks** the list into a 2-D array — a **grid** with one **row** per trial and one **column** per moment:

```
          before   start          after
trial 1   -0.1  0.1  0.0  0.1 -0.1  0.9  1.2
trial 2    0.0 -0.1  0.1  0.0 -0.1  0.8  1.0
trial 3    0.0  0.1 -0.1  0.1  0.0  1.0  0.8
```

`.shape` is now `(rows, columns)`. The comma in square brackets separates the two directions: `grid[0]` is the first row (one trial), `grid[:, 5]` is column 5 (one moment, every trial).

In [ ]:
# DEMO - every go window, stacked
go_trials = []
for start in go_starts:
    go_trials.append(dff[start - 2 : start + 5])
go_trials = np.array(go_trials)          # a list of 3 windows -> a (3, 7) grid

print(go_trials)
print("shape:          ", go_trials.shape)
print("trial 1:        ", go_trials[0])
print("column 5, all trials:", go_trials[:, 5])

### 🐞 Broken — the same loop, for nogo

The same four lines with `nogo_starts`. It crashes — the last line of the error says the rows have an **inhomogeneous shape**: they're not all the same length.

> 💡 The first nogo trial starts at frame **1**. Work out its window: `dff[1 - 2 : 1 + 5]` is `dff[-1:6]`. A negative start counts from the **end** of the array — so `dff[-1:6]` means "from frame 49 to frame 5", which is **empty**. Print it!
>
> Fix: inside the loop, only keep a trial if it has at least 2 frames before it — `if start >= 2:`.

In [ ]:
# 🐞 BROKEN — every nogo window, stacked
nogo_trials = []
for start in nogo_starts:
    nogo_trials.append(dff[start - 2 : start + 5])
nogo_trials = np.array(nogo_trials)
print(nogo_trials)


# --- self-check (do not edit) ---
assert nogo_trials.shape == (2, 7), f"❌ nogo_trials should be (2, 7) — 2 trials once the one at frame 1 is skipped — got {nogo_trials.shape}"
print("✅ Fixed — the trial at frame 1 had no 2 frames before it, so it is left out. The real data has the same problem, at frame 2.")

## Step 7 — Average down the columns: `axis=0`

One trial is noisy. But a real response happens on **every** trial, at the same moment — so **average the trials** and the noise cancels while the response stays.

`axis` says which way to collapse the grid:

| | collapses | answer | here |
|---|---|---|---|
| `.mean(axis=0)` | down the **columns** | one number per **moment** — the average trial | 7 numbers |
| `.mean(axis=1)` | along the **rows** | one number per **trial** | 3 numbers |
| `.mean()` | everything | one number | 1 number |

In [ ]:
# DEMO - the three ways to average the go grid
print("axis=0:", go_trials.mean(axis=0).round(2))    # the average trial
print("axis=1:", go_trials.mean(axis=1).round(2))    # one per trial
print("all:   ", round(go_trials.mean(), 2))

### 🔬 Write it

**Your task:**

1. Make `go_response` — the **average go trial**: one number per moment.
2. Store its highest value **from the start onwards** (position 2 on — skip the baseline), rounded to 1 decimal place, in `go_peak`.

In [ ]:
# TODO — go_response (the average trial) and go_peak


# --- self-check (do not edit) ---
assert "go_response" in globals() and "go_peak" in globals(), "❌ Make go_response and go_peak"
assert np.shape(go_response) != (3,), "❌ go_response has one value per TRIAL — that's axis=1. You want one per moment: axis=0"
assert np.shape(go_response) == (7,), f"❌ go_response should have 7 values, one per moment — go_trials.mean(axis=0). Got shape {np.shape(go_response)}"
assert go_peak == 1.0, f"❌ go_peak should be 1.0, got {go_peak} — go_response[2:].max(), rounded"
print("✅ Correct — the average go trial peaks at", go_peak, "four frames after the start")

## Step 8 — go against nogo: the answer

**Your task:**

1. Make `nogo_response` — the average **nogo** trial.
2. Store its peak from the start onwards, rounded to 1 decimal place, in `nogo_peak`.
3. Store how many times bigger the go peak is in `ratio`, rounded to 1 decimal place.

In [ ]:
# TODO — nogo_response, nogo_peak and ratio


# --- self-check (do not edit) ---
assert "nogo_response" in globals() and "nogo_peak" in globals() and "ratio" in globals(), "❌ Make nogo_response, nogo_peak and ratio"
assert np.shape(nogo_response) == (7,), f"❌ nogo_response should have 7 values — nogo_trials.mean(axis=0). Got shape {np.shape(nogo_response)}"
assert nogo_peak == 0.2, f"❌ nogo_peak should be 0.2, got {nogo_peak}"
assert ratio == 5.0, f"❌ ratio should be 5.0, got {ratio} — go_peak / nogo_peak"
print("✅ Puzzle solved — this neuron responds five times more after go than after nogo. It tells them apart.")

Here are the two answers as curves. (Plotting comes in Lab 7 — just run it.)

In [ ]:
# Run me
import matplotlib.pyplot as plt

moments = np.arange(-2, 5)        # frames from the start of the word
plt.figure(figsize=(6, 3))
plt.plot(moments, go_response, marker="o", label="go")
plt.plot(moments, nogo_response, marker="o", label="nogo")
plt.axvline(0, color="black", lw=1)
plt.xlabel("frames from the start of the word")
plt.ylabel("average activity")
plt.legend()
plt.show()

## From 50 frames to 21,238

You've just done the whole analysis. The real neuron is the **same steps** with bigger numbers:

| step | here | the real neuron |
|---|---|---|
| frames | 50 | 21,238 |
| each word lasts | 3 frames | 9 frames |
| 4 · trial starts | `go_on[::3]` | `go_on[::9]` |
| 5 · window | `dff[start - 2 : start + 5]` | `dff[start - 10 : start + 30]` |
| 6 · skip trials too near the start | `if start >= 2` | `if start >= 10` |
| 6 · grid | `(3, 7)` | `(254, 40)` |
| 7 · peak after the start | `go_response[2:]` | `go_response[10:]` |

**Next:** choose one —

- **`02_neuron_analysis_basics.ipynb`** — the same tasks as this notebook, one for one, on the real data
- **`02_neuron_analysis_advanced.ipynb`** — the real data with checkpoints but no skeletons, and stretch tasks after it

## ⏱️ Bonus — why numpy at all? (optional)

You saw this race in the lecture. Run it yourself: add 1 to **ten million** numbers — with a list and a loop, then with an array.

`%%timeit` is a notebook trick, not Python: as the **first line** of a cell it runs the cell several times and reports the average time. Watch the units: **ms** is a thousandth of a second, **µs** a millionth.

> ⏳ The first cell takes a few seconds — it is making ten million random numbers.

In [ ]:
import random

N = 10_000_000
values = [random.random() for _ in range(N)]   # a plain Python list
arr = np.array(values)                         # the same numbers, as an array
print(len(values), "numbers in the list,", arr.size, "in the array")

In [ ]:
%%timeit
# the list way: one value at a time
new_values = []
for v in values:
    new_values.append(v + 1)

In [ ]:
%%timeit
# the numpy way: every value at once
new_arr = arr + 1

Same answer — numpy usually wins by 50× or more. The loop hands Python ten million numbers one by one; numpy does the whole job in fast compiled code in one go. That is **vectorisation**: on 50 frames you can't feel it, on 21,238 frames times 254 trials you can.

## ✅ You've got it if you can…

- [ ] Say the difference between `dff.max()` and `dff.argmax()`, and between `dff.max` and `dff.max()`
- [ ] Filter an array with a mask — and explain why the mask missed the response
- [ ] Turn a mask into positions with `np.where(...)[0]`
- [ ] Explain why `[::3]` gives the start of each trial
- [ ] Cut a window round a start, and say what a negative start does
- [ ] Stack windows into a grid, and say what `.mean(axis=0)` and `.mean(axis=1)` give